# Clustering with Two Features

# Introduction

In Projects 2–5 you trained **supervised** models: every observation came
with a known label, and the algorithm learned to reproduce it. This notebook
turns that assumption off. The Survey of Consumer Finances (SCF) tells us how
much each household owes and what their home is worth — but it never tells us
which "type" of household we are looking at. That grouping is exactly what we
want to discover.

You will build your first **K-means clustering** model to segment
credit-fearful households using two financial features: household debt and home
value. Along the way you'll see how an unsupervised algorithm can surface
natural structure in data without any predefined answer key.

🎯 **By the end of this notebook you will be able to:**

-   Understand K-means clustering as an unsupervised learning algorithm.
-   Create a feature matrix for clustering from financial data.
-   Fit a K-means model and extract cluster labels and centroids.
-   Evaluate clustering quality using inertia and silhouette score.
-   Select an appropriate number of clusters using the elbow method.
-   Visualize and interpret cluster assignments in a business context.

➡️ We start with the *why* (when and why segmentation matters), then meet the
algorithm on a tiny toy dataset, and finally turn it loose on thousands of real
households.

## Watch first

The video below frames the business problem behind consumer segmentation. Keep
it in mind as a north star: every metric and plot in this notebook ultimately
serves the question *"which households resemble each other, and what should a
lender do about it?"*

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1170269319", h="3298dbabb7", width=700, height=450) 

# 1. Conceptual Foundation

## Project context: what we're trying to build

In the previous notebook you explored the SCF dataset and isolated households
that have experienced credit friction — turned down for credit or afraid of
being denied (`TURNFEAR == 1`). Now you'll take that same population and split
it into distinct groups based on financial behaviour.

💡 **Why segment consumers at all?** Financial institutions rarely treat every
customer the same way. Segmentation lets them:

-   Design targeted products (e.g., mortgage refinancing for high-debt
    households).
-   Identify risk profiles to inform credit decisions.
-   Personalize marketing and outreach instead of blasting everyone identically.

🧠 **Supervised vs. unsupervised — the key mental shift.** In Projects 2–5 a
target column (`bankrupt`, `charged_off`, …) supervised the learning. Clustering
has *no target*. The algorithm doesn't predict a known answer; it proposes
structure by grouping similar observations together. There is no "correct" label
to check against — which is exactly why evaluation works differently here.

## How K-means clustering works

K-means partitions data into *k* clusters by repeatedly nudging *k* cluster
centers (**centroids**) until each point sits as close as possible to its own
center. One pass of the algorithm looks like this:

1.  **Initialize** — randomly place *k* centroids in the feature space.
2.  **Assign** — attach each data point to its nearest centroid.
3.  **Update** — move each centroid to the mean of the points assigned to it.
4.  **Repeat** — loop steps 2–3 until the centroids stop moving (convergence).

🔄 That assign-then-update cycle is the whole engine. Because step 1 is random,
two runs can land on slightly different clusters — which is why we always pin
`random_state` for reproducibility.

📌 The number of clusters *k* is a **hyperparameter**: you must choose it before
training. Too few clusters blur real distinctions; too many fragment the data
into groups too small to act on. Section 6 gives you tools to choose *k*
deliberately instead of guessing.

## Evaluation metrics for clustering

Without true labels we can't compute accuracy or F1-score. Instead we judge
clusters by their *geometry* — how tight and how separated they are. Two metrics
do the heavy lifting:

| Metric | What it measures | Direction | Watch out for |
| --- | --- | --- | --- |
| **Inertia** | Within-cluster sum of squared distances to the centroid (compactness) | **Lower** is tighter | Always shrinks as *k* grows — never minimize it blindly |
| **Silhouette score** | Distance to own cluster vs. nearest other cluster (separation), per point, averaged | **Higher** is better, range −1 → +1 | Slow to compute on large data; a value > 0.5 signals real structure |

🔍 The two metrics answer different questions. Inertia asks *"are points close to
their own center?"* Silhouette asks *"are clusters actually distinct from each
other?"* You'll use them together: inertia for the elbow, silhouette to confirm
the elbow corresponds to genuinely separated groups.

## Demo: K-means with scikit-learn

Before touching the real data, let's watch K-means work on a tiny synthetic
dataset of eight households. With only eight points and two obvious blobs, you
can verify by eye that the algorithm finds what you'd expect — which builds
trust before we scale up to thousands of rows.

In [ ]:
# Create a small toy dataset for demonstration
import pandas as pd
from sklearn.cluster import KMeans

# Toy data: 8 households with debt and home value (in thousands)
toy_data = pd.DataFrame({
    "debt": [50, 60, 200, 220, 180, 55, 210, 65],
    "home_value": [100, 120, 400, 450, 380, 110, 420, 130]
})
print("Toy dataset:")
print(toy_data)

Now let's fit a K-means model with 2 clusters:

In [ ]:
# Instantiate and fit a KMeans model
km = KMeans(n_clusters=2, random_state=42)
km.fit(toy_data)

# Access cluster labels (which cluster each point belongs to)
print("Cluster labels:", km.labels_)

# Access cluster centroids (center of each cluster)
print("\nCentroid coordinates:")
print(km.cluster_centers_)

Read the output carefully — these two attributes are the heart of every K-means
model you'll build:

-   `labels_` is an array of integers (here `0` or `1`), one per row, naming the
    cluster each point was assigned to.
-   `cluster_centers_` is a 2D array; each row is the mean coordinate of one
    cluster — the centroid the algorithm settled on.

✅ Sanity check: the two centroids should sit roughly in the middle of the two
visible blobs (low-debt/low-value vs. high-debt/high-value).

In [ ]:
# Access inertia (within-cluster sum of squares)
print("Inertia:", km.inertia_)

**Inertia** is the single number summarizing how tight these clusters are: the
total squared distance from every point to its assigned centroid. Smaller means
more compact. On its own the raw value is hard to read — its real use comes in
Section 6, where we compare inertia *across* values of *k*.

In [ ]:
from sklearn.metrics import silhouette_score

# Calculate silhouette score
ss = silhouette_score(toy_data, km.labels_)
print(f"Silhouette score: {ss:.3f}")

A silhouette score close to **+1** indicates well-separated clusters; values
near **0** suggest clusters that overlap; negative values mean points were
likely assigned to the wrong cluster. For two clean blobs you should see a high
score, confirming the split is real and not an artifact.

## Demo: Visualizing clusters with seaborn

Numbers tell you *that* clustering happened; a scatter plot tells you *what* it
looks like. We color points by cluster with `hue` and overlay the centroids so
the structure is unmistakable:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot colored by cluster
sns.scatterplot(
    x=toy_data["debt"], y=toy_data["home_value"], hue=km.labels_, palette="deep"
)

# Overlay centroids as star markers
plt.scatter(
    km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
    color="gray", marker="*", s=200, label="Centroids"
)
plt.xlabel("Debt ($K)")
plt.ylabel("Home Value ($K)")
plt.title("Toy Data: K-Means Clustering")
plt.legend()
plt.show()

📊 In the plot above, each color is a cluster and the gray stars mark the
centroids. The two groups should separate cleanly along the diagonal — low-debt,
low-value households in one corner and high-debt, high-value households in the
other. This visual confirmation is exactly the intuition we'll carry into the
real data.

## Demo: Grouped summary statistics with pandas

A scatter plot shows the shape; a `groupby` shows the numbers behind each group.
Attaching the labels back to the data and averaging per cluster is how we
*interpret* what a cluster actually represents:

In [ ]:
# Add cluster labels to the DataFrame
toy_data["cluster"] = km.labels_

# Compute mean of each feature by cluster
cluster_summary = toy_data.groupby("cluster")[["debt", "home_value"]].mean()
print("Mean values per cluster:")
print(cluster_summary)

🔄 This pattern — **group by cluster label, then aggregate** — is the bridge from
"the model found groups" to "here is what each group means in dollars." You'll
reuse it on the real data in Section 7 to give each cluster a human-readable
financial profile.

⚠️ **Common pitfalls and debugging tips**

-   **Forgetting `random_state`** — K-means initializes randomly, so results
    drift between runs. Always pin it for reproducibility.
-   **Features on different scales** — K-means uses Euclidean distance, so a
    feature with larger magnitude dominates. Here both features are in dollars,
    so scaling is less critical; in the *next* notebook (multiple features) you
    will standardize.
-   **Choosing *k* arbitrarily** — let the elbow method and silhouette scores
    guide *k*, not visual appeal alone.
-   **Over-interpreting clusters** — clusters are analytical constructs, not
    ground truth. Validate every interpretation against domain knowledge.

✅ **Sanity checks to perform** — before and after clustering, verify:

-   **Feature matrix shape** — rows = observations, columns = selected features.
-   **No missing values** — K-means cannot handle `NaN`.
-   **Label counts** — clusters should have reasonable sizes; a cluster with a
    handful of points is usually noise.
-   **Centroid values** — centroids must fall within the range of your data.

📦 **Key points**

-   [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html):
    `n_clusters` sets the number of clusters; `random_state` ensures
    reproducibility; `labels_` returns cluster assignments;
    `cluster_centers_` returns centroid coordinates; `inertia_` returns
    within-cluster sum of squares.
-   [`silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html):
    Takes the feature matrix and labels; returns a score between −1 and +1.
-   [`pandas.DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html):
    Groups rows by a column (or array) and enables aggregate computations.
-   [`seaborn.scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html):
    Creates scatter plots; use `hue` to color points by category.

# Applied Exercises

## 2. Setup

🔧 Gather every import in one cell at the top of the notebook. Running it first
guarantees that all the names you need — plotting, pandas, the model, the metric
— are in scope before any exercise runs.

**Code 6.2.2.1**:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 3. Data Preparation

### Problem

Before building a clustering model you need clean, focused data. The SCF holds
thousands of households, but we care specifically about those who were turned
down for credit or feared denial (`TURNFEAR == 1`). You must load that subset and
select the two features that will define the clusters.

### Approach

Write a `wrangle` function that loads the compressed CSV, filters to
credit-fearful households, and returns a clean DataFrame. Then extract a feature
matrix `X` holding `DEBT` (total household debt) and `HOUSES` (primary residence
value).

📌 For definitions of these variables, see `data-dictionary.ipynb`. Note that
`wrangle` is defined right here in the notebook — unlike a shared `data.py`
module, each P6 lesson carries its own copy.

### Tasks

Define a reusable function that loads the data and keeps only rows where
`TURNFEAR == 1`.

**Code Task 6.2.3.1**:

In [ ]:
# Define a wrangle function that:
# 1. Reads the CSV file at the given filepath
# 2. Filters to rows where TURNFEAR == 1
# 3. Returns the filtered DataFrame
def wrangle(filepath):
    df = pd.read_csv(filepath)
    mask = df['TURNFEAR'] == 1
    return df[mask]


Now put `wrangle` to work and load the dataset. You should land on a DataFrame
with several thousand rows — one per credit-fearful household.

**Code 6.2.3.2**:

In [ ]:
# Use wrangle to load "data/SCFP2019.csv.gz" into df
df = wrangle("data/SCFP2019.csv.gz")
print("DataFrame shape:", df.shape)
df.head()

With the data loaded, carve out the two columns K-means will cluster on. The
result should have the same row count as `df` and exactly two columns.

**Code Task 6.2.3.3**:

In [ ]:
# Create feature matrix X with columns "DEBT" and "HOUSES"
X = df[['DEBT', 'HOUSES']]
print("Feature matrix shape:", X.shape)
X.head()

### Checkpoint

🧪 The asserts below are your contract before modeling: the data isn't empty, the
feature matrix has exactly the two expected columns in the right order, and there
are no missing values for Euclidean distance to choke on. If any line fails, fix
`wrangle` or the column selection before moving on.

In [ ]:
assert df.shape[0] > 0, "DataFrame should not be empty after filtering."
assert X.shape[1] == 2, (
    f"Feature matrix should have 2 columns, got {X.shape[1]}."
)
assert list(X.columns) == ["DEBT", "HOUSES"], (
    f"Expected columns ['DEBT', 'HOUSES'], got {list(X.columns)}."
)
assert X.isna().sum().sum() == 0, (
    "Feature matrix should have no missing values."
)
print("All checks passed!")
print(f"Working with {X.shape[0]} households and {X.shape[1]} features.")

## 4. Exploratory Visualization

### Problem

Before clustering, look at the raw relationship between your two features. A
scatter plot reveals whether natural groupings even exist — and hints at what
K-means will latch onto.

### Approach

Plot home value (`HOUSES`) against household debt (`DEBT`), scaling both to
millions of dollars so the axes stay readable.

### Tasks

Create a scatter plot of the joint distribution of debt and home value. Look for
visible clusters or patterns the algorithm might capture.

**Code 6.2.4.1**:

In [ ]:
# Create a scatter plot of HOUSES vs DEBT (scaled to millions)
sns.scatterplot(x=df["DEBT"] / 1e6, y=df["HOUSES"] / 1e6)
plt.xlabel("Household Debt [$1M]")
plt.ylabel("Home Value [$1M]")
plt.title("Credit Fearful: Home Value vs. Household Debt");

📊 Study the cloud of points before you cluster it. Most households concentrate
near the origin (modest debt, modest home value), with a sparse tail stretching
toward higher values. There's no crisp gap between groups — which foreshadows an
important lesson: K-means will *impose* boundaries on this continuum, so the
"right" number of clusters is a judgment call backed by metrics, not something
the plot hands you for free.

### Checkpoint

🧪 This checkpoint is a visual one — it simply confirms the plot rendered. The
print statements remind you what to look for rather than asserting a numeric
condition.

In [ ]:
# Visual check: The scatter plot should display points showing the
# relationship between debt and home value. Look for any apparent clusters.
print("Checkpoint: Verify the scatter plot displays correctly above.")
print("Look for natural groupings in the data that clustering might capture.")

## 5. Building the K-Means Model

### Problem

Time to segment households for real. You'll fit a K-means model, pull out the
cluster labels and centroids, and visualize the result.

### Approach

Use `KMeans` with `n_clusters=3` and `random_state=42`. After fitting, read
`labels_` for the per-household assignments and `cluster_centers_` for the
centroids, then plot points colored by cluster with the centroids overlaid.

📌 Why start with 3? It's a deliberate first guess, not the final answer.
Section 6 will test a whole range of *k* and let the metrics confirm or overturn
this choice.

### Tasks

Instantiate a 3-cluster K-means model and fit it to your feature matrix. The
assignments are learned during the `fit` call.

**Code Task 6.2.5.1**:

In [ ]:
# Build and fit a KMeans model with n_clusters=3 and random_state=42
model = KMeans(n_clusters=3, random_state=42)
model.fit(X)


Extract the cluster label assigned to each household — an array of integers
(`0`, `1`, or `2`), one per observation.

**Code 6.2.5.2**:

In [ ]:
# Extract cluster labels from the fitted model
labels = model.labels_
print("First 10 labels:", labels[:10])

Now extract the centroids. The result should be a 2D array of shape `(3, 2)` —
the mean debt and home value for each of the three clusters.

**Code 6.2.5.3**:

In [ ]:
# Extract centroids from the fitted model
centroids = model.cluster_centers_
print("Centroids shape:", centroids.shape)
print("Centroids:\n", centroids)

Visualize the segmentation: color each point by its cluster and drop the
centroids on top as star markers so you can see where each group centers.

**Code 6.2.5.4**:

In [ ]:
# Create scatter plot with points colored by cluster (use hue=labels)
# Use sns.scatterplot with x, y scaled to millions, hue=labels
sns.scatterplot(
    x=df["DEBT"] / 1e6, y=df["HOUSES"] / 1e6, hue=labels, palette="deep"
)
# Add centroids as gray stars (marker="*", s=150)
plt.scatter(
    centroids[:, 0] / 1e6, centroids[:, 1] / 1e6,
    color="gray", marker="*", s=150
)
plt.xlabel("Household Debt [$1M]")
plt.ylabel("Home Value [$1M]")
plt.title("Credit Fearful: Home Value vs. Household Debt");

📊 With three clusters the algorithm slices the debt–value cloud into bands. The
stars sit at the heart of each band, and because there's no natural gap in the
data, the boundaries fall wherever minimizing inertia places them. Ask yourself
while looking: do three groups tell a cleaner story than two or four? That's the
question Section 6 answers quantitatively.

### Checkpoint

🧪 These asserts lock in the shape of what you just built: one label per
household, exactly three centroids in two dimensions, and all three cluster IDs
present. The final print reports cluster sizes — a quick way to spot a degenerate
tiny cluster.

In [ ]:
assert labels is not None, "Labels should be extracted from the model."
assert len(labels) == X.shape[0], (
    f"Expected {X.shape[0]} labels, got {len(labels)}."
)
assert centroids.shape == (3, 2), (
    f"Expected centroids shape (3, 2), got {centroids.shape}."
)
assert set(labels) == {0, 1, 2}, (
    f"Expected labels {{0, 1, 2}}, got {set(labels)}."
)
print("All checks passed!")
print(
    f"Cluster sizes: {pd.Series(labels).value_counts().sort_index().tolist()}"
)

## 6. Evaluating Cluster Quality

### Problem

How do you know 3 clusters is the right call? You don't — yet. You need
quantitative metrics to evaluate quality and compare different values of *k*.

### Approach

Compute inertia and silhouette score for the current model, then sweep *k* from 2
to 12, recording both metrics at each step. Read the inertia curve for an
"elbow" and the silhouette curve for a peak; together they point to the best *k*.

### Tasks

Calculate the inertia and silhouette score for your 3-cluster model — compactness
and separation, side by side.

**Code 6.2.6.1**:

In [ ]:
# Extract inertia from the model and calculate silhouette score
inertia = model.inertia_
ss = silhouette_score(X, model.labels_)
print("Inertia (3 clusters):", inertia)
print("Silhouette Score (3 clusters):", ss)

These two numbers are your baseline. Inertia is large (it sums squared
*dollar* distances, so the magnitude looks huge — that's expected and fine).
Silhouette is the one to watch: hold it in mind as you sweep *k* and look for the
value that pushes it highest.

Now repeat the fit for every *k* from 2 to 12, storing both metrics so you can
compare them across the whole range.

**Code Task 6.2.6.2**:

In [ ]:
# Loop over n_clusters from 2 to 12 (inclusive)
# For each k: fit KMeans, append inertia to inertia_errors,
# append silhouette_score to silhouette_scores
n_clusters = range(2, 13)
inertia_errors = []
silhouette_scores = []

for k in n_clusters:
    temp_model = KMeans(n_clusters=k, random_state=42)
    temp_model.fit(X)
    inertia_errors.append(temp_model.inertia_)
    silhouette_scores.append(silhouette_score(X, temp_model.labels_))

print("Inertia values:", inertia_errors)
print("Silhouette scores:", silhouette_scores)

Plot inertia against the number of clusters and hunt for the **elbow** — the
point where the curve bends and extra clusters stop buying much compactness.

**Code 6.2.6.3**:

In [ ]:
# Create a line plot of inertia_errors vs n_clusters
plt.plot(n_clusters, inertia_errors)
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("K-Means Model: Inertia vs Number of Clusters");

📊 **Reading the elbow.** Inertia falls steeply at first, then flattens. The
"elbow" is where the steep drop gives way to the gentle slope: before it, each
new cluster meaningfully tightens the fit; after it, you're mostly splitting
already-tight groups. The elbow on this curve points toward roughly four
clusters — but confirm with silhouette before committing.

Now plot the silhouette score against the number of clusters. Higher is better,
so look for a clear peak.

**Code 6.2.6.4**:

In [ ]:
# Create a line plot of silhouette_scores vs n_clusters
plt.plot(n_clusters, silhouette_scores)
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("K-Means Model: Silhouette Score vs Number of Clusters");

📊 **Reading silhouette.** Where inertia only ever decreases, silhouette can rise
*and* fall — so its peak is a genuine recommendation rather than a judgment call.
When the silhouette peak lines up with the inertia elbow, you have two
independent signals agreeing on the same *k*. That agreement is what gives you
confidence to fix the final cluster count.

### Checkpoint

🧪 These asserts confirm you collected the right number of measurements (11
values, for k=2…12) and that they're in valid ranges — positive inertia,
silhouette within [−1, 1]. They guard against off-by-one loop bugs before you
read the plots.

In [ ]:
assert len(inertia_errors) == 11, (
    f"Expected 11 inertia values (k=2 to 12), got {len(inertia_errors)}."
)
assert len(silhouette_scores) == 11, (
    f"Expected 11 silhouette scores, got {len(silhouette_scores)}."
)
assert all(i > 0 for i in inertia_errors), (
    "All inertia values should be positive."
)
assert all(-1 <= s <= 1 for s in silhouette_scores), (
    "Silhouette scores should be between -1 and 1."
)
print("All checks passed!")
print("Examine the plots to identify the optimal number of clusters.")
print("Look for the 'elbow' in inertia and high silhouette scores.")

## 7. Final Model and Interpretation

### Problem

With the metrics in hand, commit to a cluster count, fit the final model, and —
the real payoff — translate each cluster into a consumer financial profile.

### Approach

The inertia elbow and the silhouette peak both favor **4 clusters**. Fit a final
`KMeans` with `n_clusters=4`, visualize the segmentation, and build a summary of
mean debt and home value per cluster to interpret who each group is.

### Tasks

Build and fit the final 4-cluster model based on your evaluation.

**Code Task 6.2.7.1**:

In [ ]:
# Build and fit final_model with n_clusters=4 and random_state=42
final_model = KMeans(n_clusters=4, random_state=42)
final_model.fit(X)


Visualize the four-cluster segmentation. Each color is now a candidate consumer
segment with its own financial signature.

**Code 6.2.7.2**:

In [ ]:
# Create scatter plot with final_model labels
# Color points by cluster using hue=final_model.labels_
sns.scatterplot(
    x=df["DEBT"] / 1e6, y=df["HOUSES"] / 1e6,
    hue=final_model.labels_, palette="deep"
)
plt.xlabel("Household Debt [$1M]")
plt.ylabel("Home Value [$1M]")
plt.title("Credit Fearful: Home Value vs. Household Debt");

📊 Four clusters carve the debt–value space into finer bands than three did. Look
for the segment hugging the axes (low debt, low home value), one or two
middle-income bands, and a sparse high-value band. The next step turns these
colors into numbers.

Compute the mean debt and home value for each cluster — the summary table that
makes interpretation concrete.

**Code 6.2.7.3**:

In [ ]:
# Create DataFrame xgb with mean DEBT and HOUSES per cluster
# Group X by final_model.labels_ and compute mean
xgb = X.groupby(final_model.labels_).mean()
xgb

📊 This little table is the deliverable a business stakeholder actually cares
about. Each row is a segment; the two columns are its average debt and average
home value. Read across the rows and a story emerges — some households carry debt
far below their home value (equity to lend against), others carry debt close to
their home value (refinancing candidates).

Finally, turn the table into a bar chart so the profiles can be compared at a
glance.

**Code 6.2.7.4**:

In [ ]:
# Create side-by-side bar chart of xgb (divided by 1e6)
(xgb / 1e6).plot(kind="bar")
plt.xlabel("Cluster")
plt.ylabel("Value [$1 million]")
plt.title("Mean Home Value & Household Debt by Cluster");

📊 Side-by-side bars make the contrast between segments obvious: clusters where
the home-value bar towers over the debt bar are equity-rich; clusters where the
two bars are close are leveraged. This single chart is what you'd put in front of
a product team to decide which segment gets which offer.

### Checkpoint

🧪 The final asserts confirm the model has 4 clusters, the summary table is the
expected `(4, 2)` shape, and — crucially — the silhouette score clears 0.5,
the rough threshold for "real" structure. Passing this is your evidence that
four clusters describe genuinely distinct household types, not arbitrary slices.

In [ ]:
assert final_model is not None, "final_model should be defined."
assert final_model.n_clusters == 4, (
    f"Expected 4 clusters, got {final_model.n_clusters}."
)
assert xgb.shape == (4, 2), f"Expected xgb shape (4, 2), got {xgb.shape}."
final_ss = silhouette_score(X, final_model.labels_)
assert final_ss > 0.5, (
    f"Silhouette score should be > 0.5 for good clustering, got {final_ss:.3f}."
)
print("All checks passed!")
print(f"Final model silhouette score: {final_ss:.3f}")
print("\nCluster summary (mean values in millions):")
print((xgb / 1e6).round(3))

# Wrap-up

In this notebook you accomplished the following:

-   Created a `wrangle` function to load and filter SCF data to credit-fearful
    households.
-   Built a feature matrix with two financial variables: household debt and home
    value.
-   Fit K-means models and extracted cluster labels and centroids.
-   Evaluated clustering quality using inertia and silhouette score.
-   Used the elbow method to select the optimal number of clusters (4).
-   Visualized clusters and interpreted them as distinct consumer profiles.

🧠 **The bigger picture.** The four clusters reveal genuinely different financial
profiles among credit-fearful households. Some show high home value relative to
debt — natural candidates for home-equity products. Others carry debt that nearly
matches their home value — candidates for mortgage refinancing. The same K-means
machinery that found two toy blobs has just produced an actionable map of a real
customer base.

➡️ **What's next.** Two features were enough to learn the mechanics, but real
segmentation rarely lives in two dimensions. In the next notebook you'll cluster
on *many* features at once — which forces two new skills: **standardizing**
features so no single dollar amount dominates the distance calculation, and using
**PCA** to compress many dimensions down to a picture you can actually see.